In [49]:
import os
import gc
import sys
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from tqdm import tqdm  # Progress bar library
import multiprocessing as mp

In [50]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

### random forest


In [51]:
def calc_random_forest(model_in, site_name, var_name):

    PLUMBER2_path_site = f"/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/{site_name}.nc"
    PLUMBER2_met_path  = "/g/data/w97/mm3972/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"
    file_met_path      = glob.glob(PLUMBER2_met_path+"/*"+site_name+"*.nc")
    
    # prepare dataset
    f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
    f_met              = nc.Dataset(file_met_path[0], mode='r')

    var_in             = pd.DataFrame(f.variables['obs_Tair'][:].data, columns=['Tair'])
    var_in['SWdown']   = f.variables['obs_SWdown'][:].data

    var_in['CO2']      = f_met.variables['CO2air'][:,0,0].data
    var_in['VPD']      = f_met.variables['VPD'][:,0,0].data
    var_in['Precip']   = f_met.variables['Precip'][:,0,0].data

    if var_name == 'NEE' and (model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE'):
        var_in[var_name] = f.variables[f'{model_in}_{var_name}'][:].data*(-1)
    else:
        var_in[var_name] = f.variables[f'{model_in}_{var_name}'][:].data


    if model_in in models_calc_LAI:
        var_in['LAI'] = read_LAI_model(site_name, model_in, model_LAI_names[model_in])
    else:
        var_in['LAI'] = read_LAI_obs(site_name, PLUMBER2_met_path)

    try: 
        var_in['SMtop1m'] = f.variables[f'{model_in}_SMtop1m'][:].data
    except: 
        var_in['SMtop1m'] = f.variables['model_mean_SMtop1m'][:].data


    # Assuming your dataframe is named 'df'
    # Example: df = pd.read_csv('your_data.csv')

    # Define the features (X) and the target (y)
    X = var_in[['Tair', 'SWdown', 'CO2', 'VPD', 'Precip', 'LAI', 'SMtop1m']]  # Predictors
    y = var_in['NEE']  # Target variable

    # Split the data into training (80%) and testing (20%) sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize the Random Forest Regressor
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, warm_start=True)

    # Train the model with tqdm to track progress
    n_trees = 100  # Number of trees in the random forest
    for i in tqdm(range(1, n_trees + 1)):
        rf_model.set_params(n_estimators=i)  # Increase the number of trees progressively
        rf_model.fit(X_train, y_train)  # Fit the model for each iteration

    # Make predictions on the test set
    y_pred = rf_model.predict(X_test)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    print(f"site is : {site_name}")
    print(f"Mean Squared Error (MSE): {mse}")
    print(f"R-squared (R2 Score): {r2}")

    # Optional: Display feature importance
    feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
    print("Feature Importances:")
    print(feature_importance.sort_values(ascending=False))
    var_in = None

In [ ]:
models_calc_LAI    = ['ORC2_r6593','ORC2_r6593_CO2','ORC3_r7245_NEE','ORC3_r8120','GFDL','SDGVM','QUINCY','NoahMPv401']

remove_site        = get_removed_site_names()
site_names, IGBP_types, clim_types, model_names = load_default_list()

# Calculate remaining sites
set_site_names     = set(site_names)
set_remove_site    = set(remove_site)
remain_sites       = set_site_names - set_remove_site
remain_sites       = list(remain_sites)

model_in  = 'obs'
var_name  = 'NEE'


for site_name in remain_sites:

    print(site_name)
    calc_random_forest(model_in, site_name, var_name)

AU-ASM


100%|██████████| 100/100 [01:24<00:00,  1.19it/s]


site is : AU-ASM
Mean Squared Error (MSE): 6.595685317576256e-11
R-squared (R2 Score): 0.8663136349182416
Feature Importances:
SWdown     0.408371
LAI        0.232481
SMtop1m    0.214197
CO2        0.049433
Tair       0.045318
VPD        0.044247
Precip     0.005952
dtype: float64
CN-Qia


100%|██████████| 100/100 [00:24<00:00,  4.06it/s]


site is : CN-Qia
Mean Squared Error (MSE): 6.662805016630419e-10
R-squared (R2 Score): 0.8950703665632662
Feature Importances:
SWdown     0.792891
Tair       0.063076
VPD        0.054275
LAI        0.042680
SMtop1m    0.040896
CO2        0.005508
Precip     0.000674
dtype: float64
DK-Sor


...

100%|██████████| 100/100 [01:32<00:00,  1.08it/s]


site is : US-UMB
Mean Squared Error (MSE): 4.842287906777642e-10
R-squared (R2 Score): 0.9141233025703484
Feature Importances:
SWdown     0.513857
LAI        0.396182
VPD        0.024910
SMtop1m    0.021731
Tair       0.021444
CO2        0.020478
Precip     0.001398
dtype: float64
ES-ES1


100%|██████████| 100/100 [01:29<00:00,  1.11it/s]


site is : ES-ES1
Mean Squared Error (MSE): 3.648655976813915e-10
R-squared (R2 Score): 0.8997133255573788
Feature Importances:
SWdown     0.787057
SMtop1m    0.055824
VPD        0.052351
Tair       0.037748
CO2        0.036778
LAI        0.029202
Precip     0.001041
dtype: float64
ZA-Kru


 80%|████████  | 80/100 [00:24<00:06,  3.26it/s]